# Pandas Part 4 — Answering Real Business Questions
This notebook uses **three cases** from different domains:

### Case 1 — Restaurant Operations
A classic restaurant tipping dataset is used to answer:
- Which tables generate the most revenue?
- When do servers earn better tips?
- How do party size, day, and meal time affect tip behavior?

### Case 2 — Market Expansion / Global Strategy
A classic international development dataset is used to answer:
- Which countries and continents are growing?
- Where is population and GDP concentrated?
- If a company wanted to expand internationally, what market patterns matter?

### Case 3 — Financial Performance / Time Series
A classic stock-price dataset is used to answer:
- Which stocks performed best?
- How should wide financial data be reshaped?
- How do we summarize performance over time?

---

## Learning objectives

By the end of this session, students should be able to:

- translate business questions into Pandas operations
- select the right metric for the question
- combine filtering, grouping, sorting, and reshaping in practical workflows
- summarize findings clearly for decision-makers
- build notebook-based analytical stories, not just disconnected code cells

## Imports and display settings

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Pandas version:", pd.__version__)

Pandas version: 3.0.0


## Load the case datasets

This notebook uses three local CSV files packaged with the session:

- `case1_restaurant_tips.csv`
- `case2_gapminder.csv`
- `case3_stocks.csv`

These are all **classic real-world datasets** widely used for teaching analytics and data science.


In [2]:
tips = pd.read_csv("case1_restaurant_tips.csv")
gap = pd.read_csv("case2_gapminder.csv")
stocks = pd.read_csv("case3_stocks.csv", parse_dates=["date"])

print("tips   :", tips.shape)
print("gap    :", gap.shape)
print("stocks :", stocks.shape)

tips   : (244, 7)
gap    : (1704, 8)
stocks : (105, 7)


---
# Restaurant Operations Analytics

## Business scenario

You are helping a restaurant manager answer practical questions such as:
- Which tables produce the most revenue?
- When do diners tip more generously?
- Does party size influence tip behavior?
- Are weekends meaningfully different from weekdays?
- Should staffing decisions differ between lunch and dinner?

This is a great case because it feels small and approachable, but still raises real operational questions.


In [ ]:
tips.head()

## Understand the table

### Discussion prompts
- What does one row represent?
- Which variables are numeric?
- Which variables are categorical?
- Which variable is the closest thing to revenue?
- Which variable should we create to better compare tipping behavior?


In [3]:
tips.info()

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   total_bill  244 non-null    float64
 1   tip         244 non-null    float64
 2   sex         244 non-null    str    
 3   smoker      244 non-null    str    
 4   day         244 non-null    str    
 5   time        244 non-null    str    
 6   size        244 non-null    int64  
dtypes: float64(2), int64(1), str(4)
memory usage: 13.5 KB


In [4]:
tips.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
total_bill,244.00,NaN,NaN,NaN,19.79,8.90,3.07,13.35,17.80,24.13,50.81
tip,244.00,NaN,NaN,NaN,3.00,1.38,1.00,2.00,2.90,3.56,10.00
sex,244,2,Male,157,NaN,NaN,NaN,NaN,NaN,NaN,NaN
smoker,244,2,No,151,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day,244,4,Sat,87,NaN,NaN,NaN,NaN,NaN,NaN,NaN
time,244,2,Dinner,176,NaN,NaN,NaN,NaN,NaN,NaN,NaN
size,244.00,NaN,NaN,NaN,2.57,0.95,1.00,2.00,2.00,3.00,6.00


## Create a more useful metric: tip percentage

Raw tip amount is useful, but not enough.

A \$5 tip on a \$20 bill is very different from a \$5 tip on a \$100 bill.

So a more meaningful measure is:

```python
tip_pct = tip / total_bill
```


In [5]:
tips["tip_pct"] = tips["tip"] / tips["total_bill"]
tips["tip_pct"].describe()

count   244.00
mean      0.16
std       0.06
min       0.04
25%       0.13
50%       0.15
75%       0.19
max       0.71
Name: tip_pct, dtype: float64

In [ ]:
tips[["total_bill", "tip", "tip_pct"]].head(10)

## First operational questions

- Which day generates the most total bill amount?
- Which day has the highest average tip percentage?
- Does dinner outperform lunch?


In [6]:
tips.groupby("day")["total_bill"].sum().sort_values(ascending=False)

day
Sat    1,778.40
Sun    1,627.16
Thur   1,096.33
Fri      325.88
Name: total_bill, dtype: float64

In [7]:
tips.groupby("day")["tip_pct"].mean().sort_values(ascending=False)

day
Fri    0.17
Sun    0.17
Thur   0.16
Sat    0.15
Name: tip_pct, dtype: float64

In [8]:
tips.groupby("time")[["total_bill", "tip", "tip_pct"]].mean().sort_values("total_bill", ascending=False)

,total_bill,tip,tip_pct
time,,,
Dinner,20.80,3.10,0.16
Lunch,17.17,2.73,0.16


### Different metrics answer different questions

Notice that:
- highest total revenue day
- highest average tip percentage day
- highest average bill time

may not all be the same.

## Group by multiple dimensions

Now answer more realistic questions:
- How does tipping vary by day and time?
- Does smoking status appear associated with different bill or tip patterns?
- Do larger parties behave differently?


In [9]:
tips.groupby(["day", "time"])["tip_pct"].mean().sort_values(ascending=False)

day   time  
Fri   Lunch    0.19
Sun   Dinner   0.17
Thur  Lunch    0.16
      Dinner   0.16
Fri   Dinner   0.16
Sat   Dinner   0.15
Name: tip_pct, dtype: float64

In [10]:
tips.groupby("smoker")[["total_bill", "tip", "tip_pct"]].mean()

,total_bill,tip,tip_pct
smoker,,,
No,19.19,2.99,0.16
Yes,20.76,3.01,0.16


In [11]:
tips.groupby("size")[["total_bill", "tip", "tip_pct"]].mean()

,total_bill,tip,tip_pct
size,,,
1,7.24,1.44,0.22
2,16.45,2.58,0.17
3,23.28,3.39,0.15
4,28.61,4.14,0.15
5,30.07,4.03,0.14
6,34.83,5.22,0.16


## Build a manager summary table

A restaurant manager may want one clear table showing:
- number of tables
- total bill volume
- average bill
- average tip percentage


In [12]:
restaurant_summary = (
    tips.groupby(["day", "time"])
    .agg(
        tables=("total_bill", "count"),
        total_sales=("total_bill", "sum"),
        avg_bill=("total_bill", "mean"),
        avg_tip_pct=("tip_pct", "mean")
    )
    .reset_index()
    .sort_values(["day", "total_sales"], ascending=[True, False])
)
restaurant_summary

,day,time,tables,total_sales,avg_bill,avg_tip_pct
0,Fri,Dinner,12,235.96,19.66,0.16
1,Fri,Lunch,7,89.92,12.85,0.19
2,Sat,Dinner,87,"1,778.40",20.44,0.15
3,Sun,Dinner,76,"1,627.16",21.41,0.17
5,Thur,Lunch,61,"1,077.55",17.66,0.16
4,Thur,Dinner,1,18.78,18.78,0.16


## Reflection

Use Pandas to answer:
1. Which combination of day and meal time has the highest total sales?
2. Which party size has the highest average tip percentage?
3. Are smokers associated with larger bills on average?
4. Which gender group leaves the higher average tip percentage?


In [13]:
# 1
restaurant_summary.sort_values("total_sales", ascending=False).head(1)

# 2
tips.groupby("size")["tip_pct"].mean().sort_values(ascending=False)

# 3
tips.groupby("smoker")["total_bill"].mean().sort_values(ascending=False)

# 4
tips.groupby("sex")["tip_pct"].mean().sort_values(ascending=False)

sex
Female   0.17
Male     0.16
Name: tip_pct, dtype: float64

## Case 1 mini-executive writeup

Write 3–5 bullet points as if you are advising the restaurant manager:
- When should staffing be strongest?
- Which customer patterns seem most profitable?
- Which metric should the restaurant track regularly?


---
# Case 2 — Market Expansion / Global Strategy

## Business scenario

Imagine you are supporting strategy for a company considering international expansion.

The company wants to understand:
- Which countries are large markets?
- Which continents have the biggest economic footprint?
- Which countries appear to be growing strongly over time?
- How should the company think about market size versus income level?

This case uses a famous global development dataset.


In [ ]:
gap.head()

## Understand the Gapminder dataset

Key columns:
- `country`
- `continent`
- `year`
- `lifeExp`
- `pop`
- `gdpPercap`

A very useful derived metric is **estimated GDP**:

```python
gdp = pop * gdpPercap
```

This is not a full economic model, but it gives a practical way to compare economic scale.


In [14]:
gap["gdp"] = gap["pop"] * gap["gdpPercap"]
gap.head()

,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num,gdp
0,Afghanistan,Asia,1952,28.80,8425333,779.45,AFG,4,"6,567,086,329.95"
1,Afghanistan,Asia,1957,30.33,9240934,820.85,AFG,4,"7,585,448,670.23"
2,Afghanistan,Asia,1962,32.00,10267083,853.10,AFG,4,"8,758,855,796.93"
3,Afghanistan,Asia,1967,34.02,11537966,836.20,AFG,4,"9,648,014,149.85"
4,Afghanistan,Asia,1972,36.09,13079460,739.98,AFG,4,"9,678,553,274.07"


In [15]:
gap.info()

<class 'pandas.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   str    
 1   continent  1704 non-null   str    
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   str    
 7   iso_num    1704 non-null   int64  
 8   gdp        1704 non-null   float64
dtypes: float64(3), int64(3), str(3)
memory usage: 119.9 KB


## Strategy questions at the continent level

- Which continents have the largest total population?
- Which continents have the largest total GDP?
- Which continents have the highest average life expectancy?


In [16]:
gap.groupby("continent")["pop"].sum().sort_values(ascending=False)

continent
Asia        30507333901
Americas     7351438499
Africa       6187585961
Europe       6181115304
Oceania       212992136
Name: pop, dtype: int64

In [17]:
gap.groupby("continent")["gdp"].sum().sort_values(ascending=False)

continent
Americas   113,778,705,062,986.22
Europe      96,999,150,708,288.95
Asia        89,984,560,308,464.72
Africa      13,044,584,494,926.51
Oceania      4,516,490,528,506.18
Name: gdp, dtype: float64

In [18]:
gap.groupby("continent")["lifeExp"].mean().sort_values(ascending=False)

continent
Oceania    74.33
Europe     71.90
Americas   64.66
Asia       60.06
Africa     48.87
Name: lifeExp, dtype: float64

### Discussion

Be careful with aggregation over all years.

If you group the full dataset without filtering a year, you are mixing many time periods together.

For strategic questions, we often want a **single snapshot year** first.


In [19]:
latest_year = gap["year"].max()
latest_year

np.int64(2007)

In [20]:
gap_latest = gap[gap["year"] == latest_year].copy()
gap_latest.head()

,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num,gdp
11,Afghanistan,Asia,2007,43.83,31889923,974.58,AFG,4,"31,079,291,948.89"
23,Albania,Europe,2007,76.42,3600523,"5,937.03",ALB,8,"21,376,411,360.04"
35,Algeria,Africa,2007,72.30,33333216,"6,223.37",DZA,12,"207,444,851,958.22"
47,Angola,Africa,2007,42.73,12420476,"4,797.23",AGO,24,"59,583,895,818.22"
59,Argentina,Americas,2007,75.32,40301927,"12,779.38",ARG,32,"515,033,625,356.57"


## Use the latest year as a decision snapshot

Now answer the same questions for the most recent year in the dataset.


In [21]:
gap_latest.groupby("continent")["pop"].sum().sort_values(ascending=False)

continent
Asia        3811953827
Africa       929539692
Americas     898871184
Europe       586098529
Oceania       24549947
Name: pop, dtype: int64

In [22]:
gap_latest.groupby("continent")["gdp"].sum().sort_values(ascending=False)

continent
Asia       20,707,949,957,614.86
Americas   19,418,085,651,710.10
Europe     14,795,499,331,554.99
Africa      2,380,485,684,001.31
Oceania       807,314,089,023.30
Name: gdp, dtype: float64

In [23]:
gap_latest.groupby("continent")["lifeExp"].mean().sort_values(ascending=False)

continent
Oceania    80.72
Europe     77.65
Americas   73.61
Asia       70.73
Africa     54.81
Name: lifeExp, dtype: float64

## Country ranking questions
- Which countries have the largest populations?
- Which countries have the largest estimated GDP?
- Which countries have the highest GDP per capita?


In [24]:
gap_latest.nlargest(10, "pop")[["country", "continent", "pop", "gdpPercap", "gdp"]]

,country,continent,pop,gdpPercap,gdp
299,China,Asia,1318683096,"4,959.11","6,539,500,929,092.31"
707,India,Asia,1110396331,"2,452.21","2,722,925,438,772.82"
1619,United States,Americas,301139947,"42,951.65","12,934,458,535,084.99"
719,Indonesia,Asia,223547000,"3,540.65","791,502,035,177.51"
179,Brazil,Americas,190010647,"9,065.80","1,722,598,680,331.38"
1175,Pakistan,Asia,169270617,"2,605.95","441,110,354,736.26"
107,Bangladesh,Asia,150448339,"1,391.25","209,311,822,133.85"
1139,Nigeria,Africa,135031164,"2,013.98","271,949,699,763.73"
803,Japan,Asia,127467972,"31,656.07","4,035,134,797,102.17"
995,Mexico,Americas,108700891,"11,977.57","1,301,973,070,171.29"


In [25]:
gap_latest.nlargest(10, "gdp")[["country", "continent", "pop", "gdpPercap", "gdp"]]

,country,continent,pop,gdpPercap,gdp
1619,United States,Americas,301139947,"42,951.65","12,934,458,535,084.99"
299,China,Asia,1318683096,"4,959.11","6,539,500,929,092.31"
803,Japan,Asia,127467972,"31,656.07","4,035,134,797,102.17"
707,India,Asia,1110396331,"2,452.21","2,722,925,438,772.82"
575,Germany,Europe,82400996,"32,170.37","2,650,870,893,900.92"
1607,United Kingdom,Europe,60776238,"33,203.26","2,017,969,309,929.46"
539,France,Europe,61083916,"30,470.02","1,861,227,940,621.40"
179,Brazil,Americas,190010647,"9,065.80","1,722,598,680,331.38"
779,Italy,Europe,58147733,"28,569.72","1,661,264,433,000.44"
995,Mexico,Americas,108700891,"11,977.57","1,301,973,070,171.29"


In [26]:
gap_latest.nlargest(10, "gdpPercap")[["country", "continent", "gdpPercap", "pop", "gdp"]]

,country,continent,gdpPercap,pop,gdp
1151,Norway,Europe,"49,357.19",4627926,"228,421,423,674.69"
863,Kuwait,Asia,"47,306.99",2505559,"118,530,454,006.19"
1367,Singapore,Asia,"47,143.18",4553009,"214,643,321,189.54"
1619,United States,Americas,"42,951.65",301139947,"12,934,458,535,084.99"
755,Ireland,Europe,"40,676.00",4109086,"167,141,167,137.84"
671,"Hong Kong, China",Asia,"39,724.98",6980412,"277,296,717,807.81"
1487,Switzerland,Europe,"37,506.42",7554661,"283,348,281,397.79"
1091,Netherlands,Europe,"36,797.93",16570613,"609,764,312,245.53"
251,Canada,Americas,"36,319.24",33390141,"1,212,704,377,996.04"
695,Iceland,Europe,"36,180.79",301931,"10,924,101,860.93"


### Discussion

A country can be:
- huge population but lower income per person
- smaller population but very high income per person
- large GDP because of scale, not necessarily wealth per capita

This is exactly why analysts must interpret more than one variable together.


## Group countries into strategy segments

A common strategic move is to create simple market segments.

For example:
- large population / high income
- large population / lower income
- smaller population / high income


In [27]:
gap_latest["pop_millions"] = gap_latest["pop"] / 1_000_000

def market_segment(row):
    if row["pop_millions"] >= 100 and row["gdpPercap"] >= 10000:
        return "Large population, higher income"
    elif row["pop_millions"] >= 100 and row["gdpPercap"] < 10000:
        return "Large population, lower income"
    elif row["pop_millions"] < 100 and row["gdpPercap"] >= 10000:
        return "Smaller population, higher income"
    else:
        return "Smaller population, lower income"

gap_latest["market_segment"] = gap_latest.apply(market_segment, axis=1)
gap_latest["market_segment"].value_counts()

market_segment
Smaller population, lower income     81
Smaller population, higher income    51
Large population, lower income        7
Large population, higher income       3
Name: count, dtype: int64

In [28]:
gap_latest.groupby("market_segment")[["pop", "gdp", "gdpPercap", "lifeExp"]].mean().sort_values("gdp", ascending=False)

,pop,gdp,gdpPercap,lifeExp
market_segment,,,,
"Large population, higher income","179,102,936.67","6,090,522,134,119.48","28,861.77",79.01
"Large population, lower income","471,055,313.43","1,814,128,422,858.27","3,718.42",65.30
"Smaller population, higher income","17,222,324.57","426,988,874,822.36","24,960.84",76.02
"Smaller population, lower income","18,987,390.40","66,202,922,661.70","3,369.79",61.03


## Time-based questions

Now use Pandas to answer trend questions:
- Which continents grew the most in total GDP over time?
- How did life expectancy change by continent?
- How do we reshape the data to compare years across continents?


In [29]:
continent_year = (
    gap.groupby(["year", "continent"])
    .agg(
        total_pop=("pop", "sum"),
        total_gdp=("gdp", "sum"),
        avg_lifeExp=("lifeExp", "mean")
    )
    .reset_index()
)
continent_year.head(12)

,year,continent,total_pop,total_gdp,avg_lifeExp
0,1952,Africa,237640501,"311,599,319,595.93",39.14
1,1952,Americas,345152446,"2,943,474,929,263.03",53.28
2,1952,Asia,1395357351,"1,125,160,167,580.96",46.31
3,1952,Europe,418120846,"2,549,140,243,979.36",64.41
4,1952,Oceania,10686006,"108,314,447,888.63",69.25
5,1957,Africa,264837738,"382,677,817,416.14",41.27
6,1957,Americas,386953916,"3,520,426,531,611.59",55.96
7,1957,Asia,1562780599,"1,559,825,258,920.21",49.32
8,1957,Europe,437890351,"3,299,685,154,207.46",66.70
9,1957,Oceania,11941976,"133,653,656,026.87",70.30


In [30]:
pd.pivot_table(
    continent_year,
    index="year",
    columns="continent",
    values="total_gdp",
    aggfunc="sum"
)

continent,Africa,Americas,Asia,Europe,Oceania
year,,,,,
1952,"311,599,319,595.93","2,943,474,929,263.03","1,125,160,167,580.96","2,549,140,243,979.36","108,314,447,888.63"
1957,"382,677,817,416.14","3,520,426,531,611.59","1,559,825,258,920.21","3,299,685,154,207.46","133,653,656,026.87"
1962,"456,813,601,791.81","4,228,826,736,052.44","1,984,516,677,395.12","4,169,540,792,857.32","164,672,906,489.34"
1967,"595,087,693,256.98","5,446,688,271,096.15","2,793,401,134,391.92","5,200,999,234,106.30","211,917,727,170.59"
1972,"783,756,582,644.10","6,703,979,470,338.80","4,104,729,661,327.21","6,560,743,881,997.31","268,224,218,454.81"
1977,"972,134,734,047.54","8,102,134,725,545.64","5,273,485,476,153.48","7,661,025,661,014.07","309,415,422,324.22"
1982,"1,146,100,854,318.17","9,082,850,208,737.97","6,416,158,647,325.22","8,384,522,312,154.25","352,354,302,760.14"
1987,"1,253,577,733,594.23","10,986,194,758,913.96","7,978,897,191,179.88","9,495,224,206,372.39","418,903,127,996.76"
1992,"1,365,362,841,367.34","12,247,495,515,582.13","10,134,316,417,028.62","10,281,097,422,156.95","472,638,359,652.21"


### Reshape the pivoted table back to long format


In [31]:
gdp_wide = pd.pivot_table(
    continent_year,
    index="year",
    columns="continent",
    values="total_gdp",
    aggfunc="sum"
)

gdp_long = gdp_wide.reset_index().melt(id_vars="year", var_name="continent", value_name="total_gdp")
gdp_long.head(12)

,year,continent,total_gdp
0,1952,Africa,"311,599,319,595.93"
1,1957,Africa,"382,677,817,416.14"
2,1962,Africa,"456,813,601,791.81"
3,1967,Africa,"595,087,693,256.98"
4,1972,Africa,"783,756,582,644.10"
5,1977,Africa,"972,134,734,047.54"
6,1982,Africa,"1,146,100,854,318.17"
7,1987,Africa,"1,253,577,733,594.23"
8,1992,Africa,"1,365,362,841,367.34"
9,1997,Africa,"1,561,205,038,824.96"


## Reflection

Use Pandas to answer:
1. Which continent had the highest total GDP in the latest year?
2. Which 5 countries had the largest estimated GDP in the latest year?
3. Which continent had the highest average life expectancy in the latest year?
4. Create a pivot table of average life expectancy by year and continent.
5. Convert that pivot table into long format.


In [32]:
# 1
gap_latest.groupby("continent")["gdp"].sum().sort_values(ascending=False)

# 2
gap_latest.nlargest(5, "gdp")[["country", "continent", "gdp"]]

# 3
gap_latest.groupby("continent")["lifeExp"].mean().sort_values(ascending=False)

# 4
lifeexp_wide = pd.pivot_table(
    gap,
    index="year",
    columns="continent",
    values="lifeExp",
    aggfunc="mean"
)
lifeexp_wide

# 5
lifeexp_long = lifeexp_wide.reset_index().melt(id_vars="year", var_name="continent", value_name="avg_lifeExp")
lifeexp_long.head()

,year,continent,avg_lifeExp
0,1952,Africa,39.14
1,1957,Africa,41.27
2,1962,Africa,43.32
3,1967,Africa,45.33
4,1972,Africa,47.45


## Case 2 mini-executive writeup

Write 4–6 bullet points as if you are advising an expansion strategy team:
- Which regions are biggest?
- Which markets look wealthy?
- Which markets look large but lower-income?
- What additional data would you want before making a real expansion decision?


---
# Case 3 — Stocks and Performance Tables

## Business scenario

You are helping an investment analyst compare stock performance over time.

The dataset contains normalized stock prices for several firms.  
This case is useful because:
- it introduces a financial time dimension
- it demonstrates wide vs long format clearly
- it gives another practical setting for Pandas summaries


In [33]:
stocks.head()

,date,GOOG,AAPL,AMZN,FB,NFLX,MSFT
0,2018-01-01,1.00,1.00,1.00,1.00,1.00,1.00
1,2018-01-08,1.02,1.01,1.06,0.96,1.05,1.02
2,2018-01-15,1.03,1.02,1.05,0.97,1.05,1.02
3,2018-01-22,1.07,0.98,1.14,1.02,1.31,1.07
4,2018-01-29,1.01,0.92,1.16,1.02,1.27,1.04


## Understand the stock dataset

Each company column is a time series.  
This means the dataset starts in **wide format**.

Wide format is common in spreadsheets and financial reports, but long format is often more convenient for:
- grouping
- plotting
- faceted comparisons


In [34]:
stocks.info()

<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    105 non-null    datetime64[us]
 1   GOOG    105 non-null    float64       
 2   AAPL    105 non-null    float64       
 3   AMZN    105 non-null    float64       
 4   FB      105 non-null    float64       
 5   NFLX    105 non-null    float64       
 6   MSFT    105 non-null    float64       
dtypes: datetime64[us](1), float64(6)
memory usage: 5.9 KB


## First business questions

- Which stock has the highest final normalized value?
- Which stock has the lowest final normalized value?
- How can we rank all stocks by end-of-period performance?


In [35]:
stocks.iloc[-1]

date    2019-12-30 00:00:00
GOOG                   1.21
AAPL                   1.68
AMZN                   1.50
FB                     1.10
NFLX                   1.54
MSFT                   1.79
Name: 104, dtype: object

In [36]:
final_values = stocks.iloc[-1].drop("date").sort_values(ascending=False)
final_values

MSFT   1.79
AAPL   1.68
NFLX   1.54
AMZN   1.50
GOOG   1.21
FB     1.10
Name: 104, dtype: object

### Note

Because the prices are normalized to start at 1.00, the ending value provides a simple relative performance measure.


## Reshape from wide to long

This is a perfect example of when `melt()` is useful.


In [37]:
stocks_long = stocks.melt(id_vars="date", var_name="ticker", value_name="normalized_price")
stocks_long.head(12)

,date,ticker,normalized_price
0,2018-01-01,GOOG,1.00
1,2018-01-08,GOOG,1.02
2,2018-01-15,GOOG,1.03
3,2018-01-22,GOOG,1.07
4,2018-01-29,GOOG,1.01
5,2018-02-05,GOOG,0.94
6,2018-02-12,GOOG,0.99
7,2018-02-19,GOOG,1.02
8,2018-02-26,GOOG,0.98
9,2018-03-05,GOOG,1.05


## Use grouped summaries in long format

Now that the stock data is tidy, we can group by `ticker`.


In [ ]:
stocks_long.groupby("ticker")["normalized_price"].agg(["min", "max", "mean"]).sort_values("max", ascending=False)

## Create a simple return summary table

We can compute:
- starting value
- ending value
- absolute change
- percentage change


In [38]:
stock_summary = (
    stocks_long.sort_values("date")
    .groupby("ticker")
    .agg(
        start_value=("normalized_price", "first"),
        end_value=("normalized_price", "last"),
        min_value=("normalized_price", "min"),
        max_value=("normalized_price", "max")
    )
)

stock_summary["absolute_change"] = stock_summary["end_value"] - stock_summary["start_value"]
stock_summary["pct_change"] = (stock_summary["end_value"] / stock_summary["start_value"]) - 1
stock_summary.sort_values("pct_change", ascending=False)

,start_value,end_value,min_value,max_value,absolute_change,pct_change
ticker,,,,,,
MSFT,1.00,1.79,0.99,1.80,0.79,0.79
AAPL,1.00,1.68,0.85,1.68,0.68,0.68
NFLX,1.00,1.54,1.00,1.96,0.54,0.54
AMZN,1.00,1.50,1.00,1.64,0.50,0.50
GOOG,1.00,1.21,0.89,1.23,0.21,0.21
FB,1.00,1.10,0.67,1.12,0.10,0.10


## Add time-based grouping features

Another useful step is extracting month from the date for monthly summaries.


In [39]:
stocks_long["month"] = stocks_long["date"].dt.to_period("M").astype(str)
stocks_long.head()

,date,ticker,normalized_price,month
0,2018-01-01,GOOG,1.00,2018-01
1,2018-01-08,GOOG,1.02,2018-01
2,2018-01-15,GOOG,1.03,2018-01
3,2018-01-22,GOOG,1.07,2018-01
4,2018-01-29,GOOG,1.01,2018-01


In [40]:
monthly_stock_avg = (
    stocks_long.groupby(["month", "ticker"])["normalized_price"]
    .mean()
    .reset_index()
)
monthly_stock_avg.head(12)

,month,ticker,normalized_price
0,2018-01,AAPL,0.99
1,2018-01,AMZN,1.08
2,2018-01,FB,0.99
3,2018-01,GOOG,1.03
4,2018-01,MSFT,1.03
5,2018-01,NFLX,1.14
6,2018-02,AAPL,0.97
7,2018-02,AMZN,1.18
8,2018-02,FB,0.95
9,2018-02,GOOG,0.98


## Build a comparison table for managers

Suppose a portfolio manager wants a compact comparison table:
- final performance
- maximum value reached
- average value


In [41]:
portfolio_summary = (
    stocks_long.groupby("ticker")
    .agg(
        avg_value=("normalized_price", "mean"),
        final_value=("normalized_price", lambda s: s.iloc[-1]),
        max_value=("normalized_price", "max"),
        min_value=("normalized_price", "min")
    )
    .sort_values("final_value", ascending=False)
)
portfolio_summary

,avg_value,final_value,max_value,min_value
ticker,,,,
MSFT,1.32,1.79,1.80,0.99
AAPL,1.14,1.68,1.68,0.85
NFLX,1.54,1.54,1.96,1.00
AMZN,1.39,1.50,1.64,1.00
GOOG,1.05,1.21,1.23,0.89
FB,0.95,1.10,1.12,0.67


## Practical questions for students

Use Pandas to answer:
1. Which stock had the best end-of-period performance?
2. Which stock reached the highest peak value?
3. Create a monthly average normalized price table by ticker.
4. Pivot that monthly summary so months are rows and tickers are columns.
5. Melt that pivot table back into long format.


In [42]:
# 1
stock_summary.sort_values("pct_change", ascending=False).head(1)

# 2
stock_summary.sort_values("max_value", ascending=False).head(1)

# 3
monthly_stock_avg.head()

# 4
monthly_stock_wide = pd.pivot_table(
    monthly_stock_avg,
    index="month",
    columns="ticker",
    values="normalized_price",
    aggfunc="mean"
)
monthly_stock_wide

# 5
monthly_stock_long = monthly_stock_wide.reset_index().melt(
    id_vars="month",
    var_name="ticker",
    value_name="avg_normalized_price"
)
monthly_stock_long.head()

,month,ticker,avg_normalized_price
0,2018-01,AAPL,0.99
1,2018-02,AAPL,0.97
2,2018-03,AAPL,0.99
3,2018-04,AAPL,0.98
4,2018-05,AAPL,1.08


---
# Capstone: Compare the three cases

## What stayed the same across all cases?

Across restaurant operations, international strategy, and stocks, the same Pandas workflow appeared again and again:

- inspect structure
- create useful variables
- filter to the right subset
- group by meaningful dimensions
- sort results for interpretation
- reshape tables when necessary
- explain results in plain language

This is exactly what makes Pandas so useful in business analytics.


## Reflection
1. the most important business question
2. the best Pandas operation used
3. the most decision-relevant metric
4. one limitation of the dataset


## Optional exercises
### Restaurant Operations
Recommend staffing and service strategy.

### Market Expansion
Recommend where a company should investigate expansion.

### Portfolio Review
Recommend which stock looked strongest and what additional evidence is needed.

You should support its conclusion with **specific Pandas-generated evidence**.


## Challenging exercises
- In the restaurant case, identify the top 10 individual tables by bill amount and discuss whether those are likely outliers.
- In the Gapminder case, compare the same top 10 GDP countries across two different years.
- In the stock case, compute month-over-month average changes by ticker.
- Create one polished summary table per case that could be pasted into slides for executives.


In [43]:
# Extension 1
tips.nlargest(10, "total_bill")[["total_bill", "tip", "tip_pct", "day", "time", "size"]]

,total_bill,tip,tip_pct,day,time,size
170,50.81,10.00,0.20,Sat,Dinner,3
212,48.33,9.00,0.19,Sat,Dinner,4
59,48.27,6.73,0.14,Sat,Dinner,4
156,48.17,5.00,0.10,Sun,Dinner,6
182,45.35,3.50,0.08,Sun,Dinner,3
102,44.30,2.50,0.06,Sat,Dinner,3
197,43.11,5.00,0.12,Thur,Lunch,4
142,41.19,5.00,0.12,Thur,Lunch,5
184,40.55,3.00,0.07,Sun,Dinner,2
95,40.17,4.73,0.12,Fri,Dinner,4


In [44]:
# Extension 2
gap_1982 = gap[gap["year"] == 1982].copy()
gap_2007 = gap[gap["year"] == 2007].copy()

gap_1982["gdp"] = gap_1982["pop"] * gap_1982["gdpPercap"]
gap_2007["gdp"] = gap_2007["pop"] * gap_2007["gdpPercap"]

top_1982 = gap_1982.nlargest(10, "gdp")[["country", "gdp"]].rename(columns={"gdp": "gdp_1982"})
top_2007 = gap_2007.nlargest(10, "gdp")[["country", "gdp"]].rename(columns={"gdp": "gdp_2007"})

top_compare = top_1982.merge(top_2007, on="country", how="outer")
top_compare

,country,gdp_1982,gdp_2007
0,Brazil,"906,717,258,453.53","1,722,598,680,331.38"
1,China,"962,691,820,907.92","6,539,500,929,092.31"
2,France,"1,104,669,186,492.24","1,861,227,940,621.40"
3,Germany,"1,725,845,977,575.61","2,650,870,893,900.92"
4,India,"605,852,264,691.60","2,722,925,438,772.82"
5,Italy,"934,957,147,512.01","1,661,264,433,000.44"
6,Japan,"2,296,143,737,891.30","4,035,134,797,102.17"
7,Mexico,"688,551,298,314.62","1,301,973,070,171.29"
8,United Kingdom,"1,027,209,400,659.14","2,017,969,309,929.46"
9,United States,"5,806,915,391,021.06","12,934,458,535,084.99"


In [45]:
# Extension 3
monthly_stock_wide = monthly_stock_avg.pivot(index="month", columns="ticker", values="normalized_price")
monthly_stock_wide.pct_change()

ticker,AAPL,AMZN,FB,GOOG,MSFT,NFLX
month,,,,,,
2018-01,NaN,NaN,NaN,NaN,NaN,NaN
2018-02,-0.01,0.09,-0.04,-0.04,0.01,0.17
2018-03,0.01,0.05,-0.03,0.00,0.01,0.12
2018-04,-0.01,-0.01,-0.03,-0.05,0.02,0.00
2018-05,0.10,0.07,0.12,0.05,0.05,0.09
2018-06,-0.00,0.06,0.04,0.04,0.02,0.14
2018-07,0.03,0.05,-0.00,0.05,0.06,-0.04
2018-08,0.12,0.07,-0.10,0.02,0.03,-0.07
2018-09,0.02,0.02,-0.07,-0.04,0.03,0.04


## Key takeaways
- Pandas is not just for toy exercises
- the same toolkit works across industries
- real analytics starts with business questions
- the value of Pandas is in turning raw rows into structured answers
